# Advanced 06 — Decentralized Identity & Trust for Multi-Agent Ecosystems

**Scenario:** agents from three organizations meet through an agent marketplace. They share no IdP. Build identity proof, portable evidence, trust establishment, delegation and local authorization without treating a DID as proof of trust.

> The notebook uses compact educational DID documents and cryptographic exercises. Production DID resolution, VC and DIDComm deployments should use conformant implementations for the chosen methods/profiles.


In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives import serialization
import base64,json,secrets,hashlib,copy
import pandas as pd
def b64(x): return base64.urlsafe_b64encode(x).decode().rstrip("=")
def ub64(x): return base64.urlsafe_b64decode(x+"="*(-len(x)%4))


## 1 — Parse a DID

In [ ]:
def parse_did(d):
    if not d.startswith("did:"): raise ValueError("not DID")
    _,method,specific=d.split(":",2)
    return method,specific
parse_did("did:web:corp.example:agents:claims")


## 2 — Method allowlist

In [ ]:
allowed_methods={"web","key"}
def method_allowed(d): return parse_did(d)[0] in allowed_methods
method_allowed("did:web:corp.example"),method_allowed("did:unknown:x")


## 3 — Create agent key

In [ ]:
priv=Ed25519PrivateKey.generate();pub=priv.public_key()
raw=pub.public_bytes(serialization.Encoding.Raw,serialization.PublicFormat.Raw)
b64(raw)


## 4 — Construct educational DID document

In [ ]:
did="did:web:corp.example:agents:claims"
doc={"id":did,
"verificationMethod":[{"id":did+"#auth-1","controller":did,"type":"Ed25519VerificationKey2020","publicKeyMultibase":"demo:"+b64(raw)}],
"authentication":[did+"#auth-1"],
"assertionMethod":[did+"#auth-1"]}
doc


## 5 — Challenge-response authentication

In [ ]:
nonce=secrets.token_bytes(32)
sig=priv.sign(nonce)
pub.verify(sig,nonce)
print("controller key proof verified")


## 6 — Verification relationship

In [ ]:
def authorized_for(doc,key_id,relationship):
    return key_id in doc.get(relationship,[])
authorized_for(doc,did+"#auth-1","authentication")


## 7 — Wrong key purpose

In [ ]:
doc2=copy.deepcopy(doc)
doc2["authentication"]=[]
print("authentication accepted?",authorized_for(doc2,did+"#auth-1","authentication"))


## 8 — Simulated resolver

In [ ]:
resolver={did:doc}
def resolve(d):
    result=resolver[d]
    if result["id"]!=d:raise ValueError("substitution")
    return result
resolve(did)["id"]


## 9 — DID document substitution attack

In [ ]:
evil={"id":"did:web:attacker.example","verificationMethod":[]}
resolver["did:web:victim.example"]=evil
try: resolve("did:web:victim.example")
except ValueError as e: print("blocked:",e)


## 10 — Service endpoint policy

In [ ]:
def endpoint_allowed(url):
    return url.startswith("https://") and not any(x in url for x in ["localhost","127.0.0.1","169.254."])
[endpoint_allowed(x) for x in ["https://partner.example/agent","http://localhost/admin","http://169.254.169.254/"]]


## 11 — Key rotation

In [ ]:
new_priv=Ed25519PrivateKey.generate();new_pub=new_priv.public_key()
new_raw=new_pub.public_bytes(serialization.Encoding.Raw,serialization.PublicFormat.Raw)
rotated=copy.deepcopy(doc)
rotated["verificationMethod"]=[{"id":did+"#auth-2","controller":did,"type":"Ed25519VerificationKey2020","publicKeyMultibase":"demo:"+b64(new_raw)}]
rotated["authentication"]=[did+"#auth-2"]


## 12 — Stale cache

In [ ]:
cached_key=doc["authentication"][0]
current_key=rotated["authentication"][0]
cached_key,current_key,cached_key==current_key


## 13 — Deactivation

In [ ]:
did_state={did:"active"}
did_state[did]="deactivated"
print("may authenticate?",did_state[did]=="active")
did_state[did]="active"


## 14 — Agent registration credential

In [ ]:
credential={"issuer":"did:web:governance.example","subject":did,
"type":"AgentRegistrationCredential","organization":"Corp","approved":True}
trusted_issuers={"did:web:governance.example"}


## 15 — Self-attestation vs third-party evidence

In [ ]:
self_claim={"issuer":did,"subject":did,"approved":True}
def enterprise_approval(c):return c["issuer"] in trusted_issuers and c.get("approved") is True
enterprise_approval(self_claim),enterprise_approval(credential)


## 16 — Peer agent

In [ ]:
partner_priv=Ed25519PrivateKey.generate()
partner_did="did:web:partner.example:agents:research"
partner_nonce=secrets.token_bytes(32)
partner_sig=partner_priv.sign(partner_nonce)
partner_priv.public_key().verify(partner_sig,partner_nonce)


## 17 — Mutual authentication

In [ ]:
def prove(private,challenge):return private.sign(challenge)
a_challenge=secrets.token_bytes(32);b_challenge=secrets.token_bytes(32)
pub.verify(prove(priv,b_challenge),b_challenge)
partner_priv.public_key().verify(prove(partner_priv,a_challenge),a_challenge)
print("mutual key control verified")


## 18 — Trust registry

In [ ]:
trust_registry={"did:web:corp.example":"trusted","did:web:partner.example":"trusted","did:web:unknown.example":"untrusted"}
def org_from_agent(d):
    # teaching mapping only
    return "did:web:"+parse_did(d)[1].split(":")[0]
trust_registry[org_from_agent(partner_did)]


## 19 — Trust is more than authentication

In [ ]:
facts={"authenticated":True,"organization_trusted":True,"agent_registered":True}
all(facts.values())


## 20 — Decentralized discovery

In [ ]:
marketplace=[
{"did":partner_did,"capabilities":["research.search"],"endpoint":"https://partner.example/agent"},
{"did":"did:web:unknown.example:agent:x","capabilities":["payment.admin"],"endpoint":"https://unknown.example/a"}]
pd.DataFrame(marketplace)


## 21 — Discovery is not trust

In [ ]:
for a in marketplace:
    a["trusted_org"]=trust_registry.get(org_from_agent(a["did"]))=="trusted"
pd.DataFrame(marketplace)


## 22 — Portable capability delegation

In [ ]:
delegation={"delegator":did,"delegate":partner_did,"resource":"claim:483",
"actions":["claim.read"],"audience":"claims-api","revoked":False}
delegation


## 23 — Attenuation

In [ ]:
parent={"actions":{"claim.read","claim.update"},"resource":"claim:483"}
child={"actions":{"claim.read"},"resource":"claim:483"}
child["actions"].issubset(parent["actions"]) and child["resource"]==parent["resource"]


## 24 — Escalation attempt

In [ ]:
evil_child={"actions":{"claim.read","claim.update","payment.approve"},"resource":"claim:483"}
evil_child["actions"].issubset(parent["actions"])


## 25 — Delegation verification

In [ ]:
def delegation_allows(d,caller,action,resource,audience):
    return (not d["revoked"] and d["delegate"]==caller and action in d["actions"]
            and d["resource"]==resource and d["audience"]==audience)
delegation_allows(delegation,partner_did,"claim.read","claim:483","claims-api")


## 26 — Delegation revocation

In [ ]:
delegation["revoked"]=True
delegation_allows(delegation,partner_did,"claim.read","claim:483","claims-api")
delegation["revoked"]=False


## 27 — Pairwise identifiers

In [ ]:
def pairwise(master,partner):
    return "did:peer:"+hashlib.sha256((master+"|"+partner).encode()).hexdigest()[:24]
pairwise(did,"partner-a"),pairwise(did,"partner-b")


## 28 — Selective disclosure

In [ ]:
all_claims={"org":"Corp","agent_class":"ClaimsAssistant","internal_risk":0.27,"approved":True}
needed={"org","approved"}
{k:v for k,v in all_claims.items() if k in needed}


## 29 — Sybil simulation

In [ ]:
sybils=[f"did:example:attacker-{i}" for i in range(1000)]
print(len(sybils),"identifiers, zero trusted organizational credentials")


## 30 — Reputation gaming

In [ ]:
ratings=[5]*100
unique_verified_orgs=1
print("high raw rating volume:",len(ratings),"independent verified sources:",unique_verified_orgs)


## 31 — Trust transitivity

In [ ]:
trust={("A","B")}
trust.add(("B","C"))
print(("A","C") in trust)


## 32 — Namespace collision

In [ ]:
external=[("partner-a","admin"),("partner-b","admin")]
compound=[f"{domain}:{name}" for domain,name in external]
compound


## 33 — Key compromise response

In [ ]:
incident={"did":did,"compromised_key":"#auth-1",
"actions":["rotate key","invalidate resolver cache","revoke delegations","reissue credentials"]}
incident


## 34 — Resolver compromise

In [ ]:
def high_assurance_resolution(requested,results):
    matching=[r for r in results if r.get("id")==requested]
    if len(matching)<2: return "INSUFFICIENT_INDEPENDENT_EVIDENCE"
    return "CONSISTENT" if matching[0]==matching[1] else "CONFLICT"
high_assurance_resolution(did,[rotated,rotated])


## 35 — Method downgrade

In [ ]:
policy={"high_risk_allowed_methods":{"web"}}
def method_ok_for_risk(d,risk):
    return risk!="high" or parse_did(d)[0] in policy["high_risk_allowed_methods"]
method_ok_for_risk("did:key:abc","high")


## 36 — OPA-style local facts

In [ ]:
policy_input={"principal":{"did":partner_did,"authenticated":True,"did_method":"web"},
"credentials":{"organization_verified":True,"agent_registered":True},
"delegation":{"valid":True,"actions":["claim.read"]},
"action":"claim.read","state":{"revoked":False,"quarantined":False}}


## 37 — Local authorization

In [ ]:
def authorize(i):
    return (i["principal"]["authenticated"] and
            i["credentials"]["organization_verified"] and
            i["credentials"]["agent_registered"] and
            i["delegation"]["valid"] and
            i["action"] in i["delegation"]["actions"] and
            not i["state"]["revoked"] and not i["state"]["quarantined"])
authorize(policy_input)


# 38 — Capstone: Multi-Agent Marketplace Trust Handshake

Implement:

```text
Discover external research agent
        ↓
method allowlist
        ↓
safe DID resolution
        ↓
verify authentication key purpose
        ↓
mutual challenge-response
        ↓
verify organization + agent credentials
        ↓
check trust registry
        ↓
create attenuated claim.read delegation
        ↓
verify audience/resource/delegate
        ↓
local OPA/Cedar policy
        ↓
ALLOW research agent to read one claim
        ↓
rotate key / revoke delegation / re-evaluate
```

Adversarial requirements:

1. reject unsupported DID methods;
2. reject substituted DID documents;
3. reject wrong verification relationships;
4. constrain service endpoints;
5. detect stale pre-rotation keys;
6. reject deactivated identities;
7. ignore self-asserted enterprise approval;
8. reject untrusted credential issuers;
9. do not grant trust because many Sybil DIDs exist;
10. do not rely on reputation alone;
11. reject delegation escalation;
12. never infer trust transitivity;
13. fail safely on resolver disagreement;
14. reject method downgrade for high-risk actions.


# Review questions

1. What does a DID identify?
2. What is the difference between DID Core and a DID method?
3. What information can appear in a DID document?
4. Why do verification relationships matter?
5. What does challenge-response prove?
6. Why is a DID not itself a trust credential?
7. What security assumptions does `did:web` inherit?
8. Why is resolver infrastructure security-sensitive?
9. How can service endpoints create SSRF risk?
10. How should key rotation affect resolver caches?
11. How do VCs complement DIDs?
12. Why is self-attestation weaker than third-party evidence?
13. What is the difference between discovery and trust?
14. How can federation complement decentralized identity?
15. Why must capability delegation attenuate authority?
16. Why do DIDs not solve Sybil attacks?
17. What are the limits of portable reputation?
18. Why is trust not automatically transitive?
19. How can pairwise identifiers improve privacy?
20. When should you *not* use decentralized identity for an agent?
